# Comparacion de cobertura nutricional para poblacion guatemalteca 

In [10]:
import sys
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()

HISTORICOS = PROJECT_ROOT / 'data' / 'raw' / 'ine' / 'cba_historicos'

CBA_RURAL_PATH = HISTORICOS / 'CBAR-2024-2026.xlsx'
CBA_URBANA_PATH = HISTORICOS / 'CBAU-2024-2026.xlsx'


In [11]:
import sys
from pathlib import Path

print("Python:", sys.executable)
print("Directorio actual:", Path.cwd())
print("\nPYTHONPATH:")
for p in sys.path:
    print(p)

Python: c:\Users\Luis Pe\Documents\GitHub\canasta-inteligente-gt\venv\Scripts\python.exe
Directorio actual: c:\Users\Luis Pe\Documents\GitHub\canasta-inteligente-gt\notebooks

PYTHONPATH:
C:\Users\Luis Pe\AppData\Local\Programs\Python\Python313\python313.zip
C:\Users\Luis Pe\AppData\Local\Programs\Python\Python313\DLLs
C:\Users\Luis Pe\AppData\Local\Programs\Python\Python313\Lib
C:\Users\Luis Pe\AppData\Local\Programs\Python\Python313
c:\Users\Luis Pe\Documents\GitHub\canasta-inteligente-gt\venv

c:\Users\Luis Pe\Documents\GitHub\canasta-inteligente-gt\venv\Lib\site-packages
C:\Users\Luis Pe\Documents\GitHub\canasta-inteligente-gt\src


In [12]:
from canasta_inteligente.application.cba_pipeline import load_catalog_pickle

catalog_path = PROJECT_ROOT / 'data' / 'processed' / 'cba_catalog.pkl'
if not catalog_path.exists():
    raise FileNotFoundError(
        f'No existe {catalog_path}. Ejecuta primero cba_pipeline.ipynb '
        'o el módulo canasta_inteligente.application.cba_pipeline.'
    )
catalog = load_catalog_pickle(catalog_path)


In [13]:
catalog.reload_aliases()

AttributeError: 'str' object has no attribute 'append'

In [ ]:
from canasta_inteligente.nutrition.evaluation._implementation import evaluacion_de_requerimientos_diarios, Peso
from canasta_inteligente.domain.persona import Persona
from statistics import mean
from canasta_inteligente.application.demo_canasta import seleccionar_catalogo_region

"""
https://www.prensalibre.com/guatemala/comunitario/por-que-los-guatemaltecos-son-los-mas-bajos-de-estatura-del-mundo/

Hombre: 19 años - 1.64m 
Mujer: 19 años - 1.49m

"""
altura_mujer_ej = 1.49
altura_hombre_ej = 1.64
mujer_ej = Persona(nombre="Mujer de referencia", edad=19, sexo='mujer', altura=altura_mujer_ej, peso=Peso.obetener_peso_de_referencia(altura_mujer_ej), naf='low')
hombre_ej = Persona(nombre="Hombre de referencia", edad=19, sexo='hombre', altura=altura_hombre_ej,peso=Peso.obetener_peso_de_referencia(altura_hombre_ej), naf='low')

req_mujer_promedio = evaluacion_de_requerimientos_diarios(mujer_ej)
req_hombre_promedio = evaluacion_de_requerimientos_diarios(hombre_ej)

ree_mujer = req_mujer_promedio['energia']['ree']
ree_hombre = req_hombre_promedio['energia']['ree']

print('====REE====')
print(f'Mujer: {ree_mujer}')
print(f'Hombre: {ree_hombre}')
print(f'Promedio: {mean([ree_mujer, ree_hombre])}')


====REE====
Mujer: 1876.03276538
Hombre: 2453.86817552
Promedio: 2164.95047045


# Comparacion rural - Agosto 2026

In [ ]:
catalog.get('arroz')

Food(id='arroz', name='ARROZ', category='CEREALES', aliases={'Arroz corriente', 'ARROZ', 'Arroz'}, price_timelines={'general': <canasta_inteligente.domain.prices.PriceTimeline object at 0x000001B6723F74D0>, 'rural': <canasta_inteligente.domain.prices.PriceTimeline object at 0x000001B661F8FD70>, 'urbana': <canasta_inteligente.domain.prices.PriceTimeline object at 0x000001B661F8FE90>}, nutrition=NutritionProfile(incap_code='13004', incap_name='ARROZ BLANCO, GRANO MEDIANO, CRUDO, S/ENRIQ.', category='CEREALES, GRANOS SECOS Y DERIVADOS', values_per_100g={'energia_kcal': 360.0, 'agua_pct': 13.0, 'fraccion_comestible_pct': 1.0, 'proteina_g': 6.61, 'grasa_total_g': 0.58, 'ag_sat_g': 0.16, 'ag_mono_g': 0.18, 'ag_poli_g': 0.16, 'colesterol_mg': 0.0, 'carbohidratos_g': 79.34, 'azucares_g': 0.12, 'fibra_dietetica_g': 1.3, 'ceniza_g': 0.58, 'calcio_mg': 9.0, 'hierro_mg': 0.8, 'magnesio_mg': 35.0, 'fosforo_mg': 108.0, 'potasio_mg': 86.0, 'sodio_mg': 1.0, 'zinc_mg': 1.16, 'cobre_mg': 0.11, 'selenio_

In [ ]:
from canasta_inteligente.application.demo_canasta import seleccionar_catalogo_region
import unicodedata


dias = 31
catalog_rural = seleccionar_catalogo_region(catalog, 'rural')

cba_rural = pd.read_excel(CBA_RURAL_PATH, sheet_name='Histórico producto CBAR')
cba_rural_last_canasta = cba_rural[(cba_rural['Año'] == 2026) & (cba_rural['Mes'] == 'Agosto')]
cba_rural_last_canasta['slug'] = cba_rural_last_canasta['Producto'].apply(lambda product_name: "".join(char for char in unicodedata.normalize("NFD", str.lower(product_name).replace(" ", "_")) if unicodedata.category(char) != "Mn") )
print(cba_rural_last_canasta['Producto'])


1860                                      Arroz corriente
1861                                      Arroz precocido
1862                                          Maíz blanco
1863                                       Harina de maíz
1864              Harina para atoles (incluye Incaparina)
1865                                          Pan francés
1866                                            Pan dulce
1867                                      Galletas dulces
1868                                    Tortillas frescas
1869                                           Avena/mosh
1870                                            Espagueti
1871    Fideos en todas sus formas (Excepto macarrones...
1872                               Frijoles negros, secos
1873    Frijoles preparados, procesados y condimentado...
1874                        Repollo, fresco o refrigerado
1875                      Cilantro, perejil y hierbabuena
1876                            Macuy/Hierba mora/Quilete
1877          

In [5]:
food_id = catalog_rural.resolve_by_alias('arroz_corriente')

NameError: name 'catalog_rural' is not defined

In [ ]:
cba_rural_last_canasta['Kilocalorías diarias'].sum()

In [ ]:
costo_mensual_real = cba_rural_last_canasta['Costo mensual'].sum()
costo_diario_real = cba_rural_last_canasta['Costo diario'].sum()
costo_diario_real

In [ ]:
req = hombre_ej.calcular_requerimientos()
resultados_hombre_ej = hombre_ej.calculo_canasta_diaria_costo_minimizado(catalog)
resultado_min = float('inf')
resultado_optimo = None
for resultado in resultados_hombre_ej:
    if resultado['costo_total_q'] < resultado_min:
        resultado_min = resultado['costo_total_q']
        resultado_optimo = resultado

resultado_optimo

# Exploración inicial de ENIGH

Este notebook explora las bases de **personas** y **hogares** de ENIGH y utiliza los metadatos de los archivos `.sav` para interpretar las variables codificadas.

El objetivo inicial es entender la estructura de los datos antes de construir el generador de hogares.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_DIR = PROJECT_ROOT / 'data' / 'raw' / 'ine' / 'enigh'

PERSONAS_XLSX = DATA_DIR / "enigh_personas.xlsx"
HOGARES_XLSX = DATA_DIR / "enigh_hogares.xlsx"

PERSONAS_SAV = DATA_DIR / "ENIGH_Personas_20232710.sav"
HOGARES_SAV = DATA_DIR / "ENIGH_Hogares_20232710.sav"

## 1. Carga de las bases

In [ ]:
personas = pd.read_excel(PERSONAS_XLSX)
hogares = pd.read_excel(HOGARES_XLSX)

print("Personas:", personas.shape)
print("Hogares :", hogares.shape)

In [ ]:
display(personas.head())
display(hogares.head())

In [ ]:
!pip install pyreadstat

In [ ]:
import pyreadstat

_, meta_personas = pyreadstat.read_sav(PERSONAS_SAV, metadataonly=True)
_, meta_hogares = pyreadstat.read_sav(HOGARES_SAV, metadataonly=True)

print("Variables personas:", len(meta_personas.column_names))
print("Variables hogares :", len(meta_hogares.column_names))

In [ ]:
def construir_diccionario(meta):
    labels = dict(zip(meta.column_names, meta.column_labels))
    value_labels = meta.variable_value_labels

    rows = []
    for variable in meta.column_names:
        rows.append({
            "variable": variable,
            "descripcion": labels.get(variable),
            "valores": value_labels.get(variable)
        })

    return pd.DataFrame(rows)

dic_personas = construir_diccionario(meta_personas)
dic_hogares = construir_diccionario(meta_hogares)

display(dic_personas.loc[dic_personas['valores'].isna()].head(20))

In [ ]:
def buscar_variables(diccionario, texto):
    mask = (
        diccionario["variable"].astype(str).str.contains(texto, case=False, na=False)
        | diccionario["descripcion"].astype(str).str.contains(texto, case=False, na=False)
    )
    return diccionario.loc[mask]

buscar_variables(dic_personas, "edad")

In [ ]:
buscar_variables(dic_personas, "sexo")

In [ ]:
buscar_variables(dic_personas, "parentesco")

In [ ]:
identificacion = ["DEPTO", "MUPIO", "AREA", "HOGAR", "PONDERADOR"]

display(personas[identificacion + ["ID"]].head(15))
display(hogares[identificacion].head())

In [ ]:
for col in identificacion:
    print(
        col,
        "| personas:", personas[col].nunique(dropna=False),
        "| hogares:", hogares[col].nunique(dropna=False)
    )

In [ ]:
hogar_cols = ["DEPTO", "MUPIO", "AREA", "HOGAR"]

personas["hogar_id"] = (
    personas[hogar_cols]
    .astype(str)
    .agg("-".join, axis=1)
)

hogares["hogar_id"] = (
    hogares[hogar_cols]
    .astype(str)
    .agg("-".join, axis=1)
)

print("Hogares distintos en personas:", personas["hogar_id"].nunique())
print("Filas de hogares:", len(hogares))
print("hogar_id únicos en hogares:", hogares["hogar_id"].is_unique)

## 6. Tamaño de los hogares

In [ ]:
tamano_hogar = (
    personas.groupby("hogar_id")
    .size()
    .rename("personas")
)

display(tamano_hogar.describe())

In [ ]:
tamano_hogar.value_counts().sort_index().to_frame("cantidad_hogares")

In [ ]:
hogar_ejemplo = personas["hogar_id"].iloc[0]

columnas_base = [
    "hogar_id", "ID", "DEPTO", "MUPIO", "AREA",
    "HOGAR", "PONDERADOR"
]

display(
    personas.loc[
        personas["hogar_id"] == hogar_ejemplo,
        columnas_base + ["B1P00A02", "B1P00A03", "B1P00A05"]
    ]
)

In [ ]:
def describir_variable(df, diccionario, variable):
    info = diccionario.loc[diccionario["variable"] == variable]

    if info.empty:
        print("Variable no encontrada:", variable)
        return

    display(info)

    print("\nFrecuencias:")
    display(
        df[variable]
        .value_counts(dropna=False)
        .sort_index()
        .to_frame("n")
    )

describir_variable(personas, dic_personas, "B1P00A03")

In [ ]:
describir_variable(personas, dic_personas, "B1P00A02")

In [ ]:
describir_variable(personas, dic_personas, "B1P00A05")

## 9. Cobertura y valores faltantes

Primero se revisan las variables con mayor y menor disponibilidad antes de seleccionar las que usará el generador.

In [ ]:
resumen_personas = pd.DataFrame({
    "tipo": personas.dtypes.astype(str),
    "no_nulos": personas.notna().sum(),
    "faltantes": personas.isna().sum(),
    "pct_faltantes": personas.isna().mean().mul(100).round(2),
    "unicos": personas.nunique(dropna=True)
})

display(resumen_personas.sort_values("pct_faltantes").head(25))

In [ ]:
display(
    resumen_personas
    .sort_values("pct_faltantes", ascending=False)
    .head(25)
)

## 10. Tabla compacta del diccionario

Se combinan metadatos y estadísticas observadas para facilitar la revisión.

In [ ]:
def resumen_diccionario(df, diccionario):
    estadisticas = pd.DataFrame({
        "variable": df.columns,
        "tipo": [str(df[c].dtype) for c in df.columns],
        "no_nulos": [df[c].notna().sum() for c in df.columns],
        "unicos": [df[c].nunique(dropna=True) for c in df.columns],
        "pct_faltantes": [round(df[c].isna().mean() * 100, 2) for c in df.columns],
    })

    return diccionario.merge(estadisticas, on="variable", how="right")

catalogo_personas = resumen_diccionario(personas, dic_personas)
catalogo_hogares = resumen_diccionario(hogares, dic_hogares)

display(catalogo_personas.head(30))

## 11. Variables candidatas para el generador

La selección final debe hacerse usando las etiquetas del `.sav`, no por inferencia basada únicamente en los códigos.

In [ ]:
terminos = [
    "edad",
    "sexo",
    "parentesco",
    "embar",
    "lact",
    "peso",
    "talla",
    "altura"
]

candidatas = pd.concat(
    [buscar_variables(dic_personas, termino) for termino in terminos],
    ignore_index=True
).drop_duplicates("variable")

display(candidatas)

## 12. Relación personas-hogares

Se verifica cuántos registros de personas encuentran su hogar correspondiente.

In [ ]:
merge_test = personas[["hogar_id"]].merge(
    hogares[["hogar_id"]],
    on="hogar_id",
    how="left",
    indicator=True
)

merge_test["_merge"].value_counts()

In [ ]:
from canasta_inteligente.nutrition.evaluation._implementation import evaluacion_de_requerimientos_diarios, Peso
from canasta_inteligente.domain.persona import Persona
Peso.obetener_peso_de_referencia(1.49)